In [168]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def get_tracking(particles, data, chromaticity, track): 
    filename = f"{data}"
    names = ["i", "lattice","optimization", "SGx1", "SGx2", "SGy1", "chrom_x", "chrom_y", "quad_kx", "quad_ky", "quad_kz", "obj"]
    df = pd.read_csv(filename, comment="#", sep='\\s+', header=None, names=names)
   # df = df.sort_values("N")
    lattice = df["lattice"].values
    SGx1 = df["SGx1"].values
    SGx2 = df["SGx2"].values
    SGy1 = df["SGy1"].values
    quad_kx = df["quad_kx"].values
    quad_ky = df["quad_ky"].values
    quad_kz = df["quad_kz"].values
    chrom_x = df["chrom_x"].values
    chrom_y = df["chrom_y"].values
    objectives = df["obj"].values

    lattice_name = lattice[chromaticity]
    match (lattice_name):
        case ("electrostatic"): var = ["electrostatic", 1.24810736, 112.464392, "protons", 540, 419.10]
        case ("magnetic_2p"): var = ["magnetic_2p", 1.143914, 0, "deuterons", 100, 141.014]
    
    Sext_x1 = SGx1[chromaticity]
    Sext_x2 = SGx2[chromaticity]
    Sext_y1 = SGy1[chromaticity]
    q_kx = quad_kx[chromaticity]
    q_ky = quad_ky[chromaticity]
    q_kz = quad_kz[chromaticity]
    chr_x = chrom_x[chromaticity]
    chr_y = chrom_y[chromaticity]
    obj = objectives[chromaticity]
    
    chromaticity_x = str(chrom_x[chromaticity]).replace('.', '')   # '314'
    chromaticity_y = str(chrom_y[chromaticity]).replace('.', '')   # '314'
    
    match (track):
        case ("el_by_el"):
            tracking_code1 = f"""
                INCLUDE '{var[0]}_maps';
            """
        case ("dynap"):
            tracking_code1 = f"""
                INCLUDE '{var[0]}';
            """
    tracking_code2 = f"""
    PROCEDURE INJECT1 NUM;
    VARIABLE X 100; VARIABLE I 1; VARIABLE PSI 1; VARIABLE SY 1; VARIABLE SZ 1; VARIABLE PSI0_DEG 1;
    X := LINSPACE(-{particles[1]}, {particles[1]*2}, NUM);
    CR;
    PSI0_DEG := 0;
    PSI := DEG2RAD(PSI0_DEG);
    LOOP I 1 NUM;
    SY := SIN(PSI); SZ := COS(PSI);
    SR X|I 0 0 0 0 0 0 0 1; SSR 0 SY SZ;
    SR 0 0 X|I 0 0 0 0 0 1;  SSR 0 SY SZ;
    SR 0 0 0 0 0 (X|I)/10 0 0 1;  SSR 0 SY SZ;

    SR X|I 0 X|I 0 0 0 0 0 1;  SSR 0 SY SZ;
    
    SR X|I 0 X|I 0 0 (X|I)/10 0 0 1; SSR 0 SY SZ;

  ENDLOOP;
    ENDPROCEDURE;
    
    PROCEDURE RUN;
      VARIABLE WHERE 100;
      VARIABLE MAPPARAMS 1 6; 
      VARIABLE MAPARR 1000 6 666; VARIABLE SPNRARR 1000 3 3 666;
      VARIABLE SGx1 1; VARIABLE SGy1 1; VARIABLE SGx2 1; VARIABLE SGy2 1; VARIABLE EB1 1;
      VARIABLE GAMMA 1; VARIABLE NUM_ELE 1;
     VARIABLE MRKR 1000; VARIABLE MU_N_ARR 5000 1000;
    VARIABLE MU 800; VARIABLE NBAR 800 3; VARIABLE MU_TP 800 3; VARIABLE MU0 1;
    
      OV 3 3 0;
    daeps 1e-12;
      GAMMA := {var[1]};
    NUM_ELE := {var[4]};
      SET_FOR_{var[3]} GAMMA;
      EB1 := {var[2]};
    

      SGx1 :=  {Sext_x1}; SGy1  := {Sext_y1}; 
      SGx2 :=  {Sext_x2}; SGy2  := 0.0;
      
      MRKR := '{var[0]}_{track}_chromx_{chromaticity_x}_chromy_{chromaticity_y}';
      
    """
    match (track):
        case ("el_by_el"):
            tracking_code3 = f"""
            OPENF 924 'C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/cosy/tracking data/'&MRKR&'.dat' 'REPLACE';
            LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1 MAPARR SPNRARR;
            INJECT1 {particles[0]};
          TREL MAPARR SPNRARR 1 {var[4]} 100 924 925;
              CLOSEF 924; 
                ENDPROCEDURE;
                RUN; END;
                """
            tracking_code4 = " "
        case ("dynap"):
            tracking_code3 = f"""
            LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
            catr MAP MAP {var[5]} ;
            INJECT1 {particles[0]};
            OPENF 924 'C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/cosy/tracking data/'&MRKR&'.dat' 'REPLACE';
            TRPRAY 924;
            TR {particles[4]} {particles[4]} 1 2 20 20 0 0 -1;
            CLOSEF 924; 
            OPENF 8931 'C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/cosy/tracking data/TRPSPI_'&MRKR&'.dat' 'REPLACE';
            TSS MU NBAR 0;
          GET_TUNE_ENSEMBLE MU_N_ARR;"""
            spin_tune_lines = []
            for j in range(1+particles[0]*5): 
                spin_tune_line = f"MU_N_ARR(1)|{j+1}-CONS(MU)"
                spin_tune_lines.append(spin_tune_line)
            st_line = " ".join(spin_tune_lines)
            tracking_code4 = f"""WRITE 8931 '#SPIN-TUNE'; WRITE 8931 '#SPIN-TUNE' {st_line};
            CLOSEF 8931;
            ENDPROCEDURE;
            
            RUN; END;
                    """
            
    with open(f"{var[0]}_{track}_chromx_{chromaticity_x}_chromy_{chromaticity_y}.fox", "w") as f:
        f.write(tracking_code1)
        f.write(tracking_code2)
        f.write(tracking_code3)
        f.write(tracking_code4)
        
    name = f"{var[0]}_{track}_chromx_{chromaticity_x}_chromy_{chromaticity_y}"
    all_data = [chromaticity, name, lattice_name, Sext_x1, Sext_x2, Sext_y1, chr_x, chr_y, q_kx, q_ky, q_kz, obj ]
    
    return (all_data)

In [194]:
data_and_filenames = []

chromaticities_data = "C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/COSY/chrom_optimization.dat"
with open(chromaticities_data, "r") as f:
    lines = f.readlines()

#particles: Particle number, max x/y/delta deviation, Nturn
particles = [3, 5e-3, 0, 0, 1000]
track = "dynap"

for chromaticity in range(1, len(lines)):
    c = get_tracking(particles, chromaticities_data, chromaticity, track)
    data_and_filenames.append(c)

print("нужно трекать эти файлы:")
for i in range(len(lines)-1):
    print("index=", data_and_filenames[i][0]-1, data_and_filenames[i][1])
 

нужно трекать эти файлы:
index= 0 magnetic_2p_dynap_chromx_-1879528e+00_chromy_-2637587e+00
index= 1 magnetic_2p_dynap_chromx_-1468708e-07_chromy_1325588e-06
index= 2 magnetic_2p_dynap_chromx_1245162e-06_chromy_-6819141e+00
index= 3 electrostatic_dynap_chromx_-9170757e+00_chromy_-8816105e+00
index= 4 electrostatic_dynap_chromx_-4362987e-07_chromy_-6059580e-07
index= 5 electrostatic_dynap_chromx_-1085244e+01_chromy_-8970429e+00


In [195]:
import subprocess
index = 5
c = data_and_filenames[index][1]
subprocess.Popen(f'start cmd /c cosy {c}', shell=True)

<Popen: returncode: None args: 'start cmd /c cosy electrostatic_dynap_chromx...>

In [275]:
import os

filepath = "C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/cosy/tracking data/"

def rewrite(data_and_filenames):
    for idx in range(len(data_and_filenames)):
        name = data_and_filenames[idx][1]

        infile = f"{filepath}TRPSPI_{name}.dat"
        outfile = f"{filepath}TRPSPI_{name}.dat"

        with open(infile, "r") as f:
            lines = f.readlines()

        header = "#N\tspin_tunes\n"

        with open(outfile, "w") as f:
            f.write(header)

            # пропускаем первые 2 строки
            for n, line in enumerate(lines[2:], start=1):
                value = line.strip()  # убираем лишние пробелы
                f.write(f"{n:<6}\t{value}\n")

rewrite(data_and_filenames)

In [277]:
import pandas as pd

filepath = "C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/cosy/tracking data/"
def get_lengheting(data_and_filenames, variables, f):
    filenames = [f"{filepath}/{data_and_filenames[f][1]}.dat", f"{filepath}/TRPSPI_{data_and_filenames[f][1]}.dat"]
    data = []
    for e in range (len(filenames)):
        if e==0:
            names = ["iteration", "ray", "X", "A", "Y", "B", "L", "D"]
        elif e==1:
            names = ["N", "delta_spin"]
        df = pd.read_csv(filenames[e], comment="#", sep='\\s+', header=None, names=names)
        if "iteration" in df.columns and "ray" in df.columns:
            df = df.sort_values(["iteration","ray"])
        elif "N" in df.columns:
            df = df.sort_values("N")
        data.append(df)
    Nn = data[1]["N"].values
    delta_spin = data[1]["delta_spin"].values
    x = data[0]["X"].values
    y = data[0]["Y"].values
    delta = data[0]["D"].values
    L = data[0]["L"].values
    ray = data[0]["ray"].values
    iteration = data[0]["iteration"].values
    if data_and_filenames[f][2] == variables[2]:
            betax = variables[0]
            betay = variables[1]
    if data_and_filenames[f][2] == variables[5]:
            betax = variables[3]
            betay = variables[4]

    theoretical_lenghtening = []
    cosy_lenghtening = []
    rays = []
    lenghtening = []
    lattice_data = [data_and_filenames[f][2], data_and_filenames[f][3], data_and_filenames[f][4], data_and_filenames[f][5], data_and_filenames[f][6], data_and_filenames[f][7], data_and_filenames[f][8], data_and_filenames[f][9],data_and_filenames[f][10],data_and_filenames[f][11]]
    full_data = []
    for i in range(len(Nn)):
        rays.append((ray[i], x[i], y[i], delta[i], delta_spin[i]))
    for i in range(len(iteration)):
        if iteration[i] == 0:
            theor_L = -np.pi*(float(data_and_filenames[f][6])*((x[i])**2/betax) + float(data_and_filenames[f][7])*((y[i])**2/betay))
            theoretical_lenghtening.append(theor_L)
        if iteration[i] == 1000:
            cosy_L = L[i]/iteration[i]
            cosy_lenghtening.append(cosy_L)
    full_data = [lattice_data, rays, theoretical_lenghtening, cosy_lenghtening]
    return(full_data)
# betas for each in file lattice type
variables = [21.7051, 7.75695, "electrostatic", 2.98732, 35.1547, "magnetic_2p"]
all_data = []
for n in range(len(data_and_filenames)):
    a = get_lengheting(data_and_filenames, variables, n)
    all_data.append(a)

In [300]:
print(all_data[0][2][1]) #это данные для луча 0
print(all_data[0][2][2])

4.941478781161975e-05
5.8926857165641745e-06


In [240]:
headers = ["lattice","SGx1","SGx2","SGy1","chrom_x","chrom_y","quad_kx","quad_ky","quad_kz","obj", "ray_id", "x", "y", "delta", "delta_spin", "theor_lenght", "cosy_lenghr"]
for i in range(len(f)):
    for j in range(len(f[i][1])):
        lattice = all_data[i][0][0]
        SGx1 = all_data[i][0][1]
        SGx2 = all_data[i][0][2]
        SGy1 = all_data[i][0][3]
        chrom_x = all_data[i][0][4]
        chrom_y = all_data[i][0][5]
        quad_kx = all_data[i][0][6]
        quad_ky = all_data[i][0][7]
        quad_kz = all_data[i][0][8]
        obj = all_data[i][0][9]
        ray_id = all_data[i][1][j][0]
        x = all_data[i][1][j][1]
        y = all_data[i][1][j][2]
        delta = all_data[i][1][j][3]
        delta_spin = all_data[i][1][j][4]
        theor_lenght = all_data[i][2][j]
        cosy_lenght = all_data[i][3][j]

magnetic_2p


In [315]:
headers = [
    "lattice","SGx1","SGx2","SGy1","chrom_x","chrom_y",
    "quad_kx","quad_ky","quad_kz","obj",
    "ray_id","x","y","delta","delta_spin",
    "theor_length","cosy_length"
]

col_width = 18
outfile = "final_table.dat"

with open(outfile, "w") as f_out:
    # 🔹 заголовок
    header_line = "".join(f"{h:<{col_width}}" for h in headers)
    f_out.write(header_line + "\n")

    # 🔹 данные
    for base, rays, theor, cosy in all_data:
        for j in range(len(theor)):
            row = [
                base[0],          # строка
                base[1], base[2], base[3], base[4], base[5],
                base[6], base[7], base[8], base[9],
                rays[j][0],       # ray_id
                rays[j][1],       # x
                rays[j][2],       # y
                rays[j][3],       # delta
                rays[j][4],       # delta_spin
                theor[j],
                cosy[j],
            ]

            # 🔹 форматируем все колонки одной шириной
            formatted = []
            for val in row:
                # если строка — оставляем как есть, иначе форматируем число
                if isinstance(val, str):
                    formatted.append(f"{val:<{col_width}}")
                else:
                    formatted.append(f"{float(val):>{col_width}.6e}")

            f_out.write("".join(formatted) + "\n")